In [2]:
import scanpy as sc
import anndata
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import scvi
import seaborn as sb

In [3]:
os.chdir("/home/lixiangyu/zr/Annotate/")

##### output data:/home/lixiangyu/zr/Annotate/output_data/public_data/Mouse_all/Mouse_merged.h5ad

# AS

## GSE239591 29839 27094/28114

In [3]:
file_paths = [
    "data/Mouse_public_data/GSE239591/GSE239591_ABT_f.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_ABT_M.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_Ctrl_F.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_Ctrl_M.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_HFD_F.h5ad",
    "data/Mouse_public_data/GSE239591/GSE239591_HFD_M.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]

In [4]:
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)


Number of cells in sample Sample1: 6084
Number of cells in sample Sample2: 4888
Number of cells in sample Sample3: 5469
Number of cells in sample Sample4: 5775
Number of cells in sample Sample5: 2635
Number of cells in sample Sample6: 4988


In [5]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

adata_final.write_h5ad("data/Mouse_public_data/GSE239591/GSE239591_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [7]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE239591/GSE239591_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE239591/GSE239591_merged_postR.h5ad")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [8]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE239591/GSE239591_merged_postR.h5ad")

In [9]:
adata_final

AnnData object with n_obs × n_vars = 29839 × 32288
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239591_1_UMAP', 'decontX_GSE239591_2_UMAP', 'decontX_GSE239591_3_UMAP', 'decontX_GSE239591_4_UMAP', 'decontX_GSE239591_5_UMAP', 'decontX_GSE239591_6_UMAP'
    layers: 'decontXcounts'

In [10]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [11]:
mt_genes_lower

['mt-Nd1',
 'mt-Nd2',
 'mt-Co1',
 'mt-Co2',
 'mt-Atp8',
 'mt-Atp6',
 'mt-Co3',
 'mt-Nd3',
 'mt-Nd4l',
 'mt-Nd4',
 'mt-Nd5',
 'mt-Nd6',
 'mt-Cytb']

In [12]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [13]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGAATCGCG-1               1388        3271.0       2.170590
AAACCCAAGACCTCCG-1               1995       10631.0       0.338632
AAACCCAAGAGATCGC-1               1506        4098.0       3.562714
AAACCCAAGGTAAAGG-1               4649       18316.0       1.676130
AAACCCACAGACGCTC-1               2017        5531.0       3.869101

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [14]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=6000) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 25] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 29839


Number of cells beforeMT filter: 28508
Number of cells after MT filter: 28114


In [17]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

AnnData object with n_obs × n_vars = 28114 × 24751
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239591_1_UMAP', 'decontX_GSE239591_2_UMAP', 'decontX_GSE239591_3_UMAP', 'decontX_GSE239591_4_UMAP', 'decontX_GSE239591_5_UMAP', 'decontX_GSE239591_6_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [18]:
adata_final.write("data/Mouse_public_data/GSE239591/GSE239591_postQC-R.h5ad")

In [19]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE239591/GSE239591_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 28114 × 24751
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239591_1_UMAP', 'decontX_GSE239591_2_UMAP', 'decontX_GSE239591_3_UMAP', 'decontX_GSE239591_4_UMAP', 'decontX_GSE239591_5_UMAP', 'decontX_GSE239591_6_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [20]:
adata_final.obs['sample'].value_counts()

sample
GSE239591_1    5538
GSE239591_4    5531
GSE239591_3    5241
GSE239591_2    4669
GSE239591_6    4669
GSE239591_5    2466
Name: count, dtype: int64

## GSE269449 12370 10158/10513

In [21]:

file_paths = [
    "data/Mouse_public_data/GSE269449/GSE269449_LXR.h5ad",
    "data/Mouse_public_data/GSE269449/GSE269449_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7878
Number of cells in sample Sample2: 4492


In [22]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE269449/GSE269449_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [23]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [24]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE269449/GSE269449_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE269449/GSE269449_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 17:41:43 2026 .. Analyzing cells in batch 'GSE269449_2'
Thu Apr 23 17:41:43 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 17:42:17 2026 .... Estimating contamination
Thu Apr 23 17:42:23 2026 ...... Completed iteration: 10 | converge: 0.03531
Thu Apr 23 17:42:29 2026 ...... Completed iteration: 20 | converge: 0.01225
Thu Apr 23 17:42:35 2026 ...... Completed iteration: 30 | converge: 0.006562
Thu Apr 23 17:42:41 2026 ...... Completed iteration: 40 | converge: 0.004124
Thu Apr 23 17:42:47 2026 ...... Completed iteration: 50 | converge: 0.00309
Thu Apr 23 17:42:53 2026 ...... Completed iteration: 60 | converge: 0.002666
Thu Apr 23 17:42:59 2026 ...... Completed iteration: 70 | converge: 0.001866
Thu Apr 23 17:43:05 2026 ...... Completed iteration: 80 | converge: 0.001559
Thu Apr 23 17:43:12 2026 ...... Completed iteration: 90 | converge: 0.0013

In [25]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE269449/GSE269449_merged_postR.h5ad")

In [26]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [27]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [28]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGCCTATTG-1               2236        6313.0       1.457310
AAACCCAAGGGTTGCA-1                296         908.0       2.643172
AAACCCACAAAGCGTG-1               1699        3611.0       3.461645
AAACCCACACGACAGA-1               3252        9508.0       2.166597
AAACCCACACGACCTG-1               6047       27013.0       1.003221

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [29]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
# sc.pp.filter_cells(adata_final, min_genes=200)  
sc.pp.filter_cells(adata_final, max_genes=7500) 
sc.pp.filter_genes(adata_final, min_cells=3) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 40000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 12370
Number of cells before counts filter: 12335
Number of cells beforeMT filter: 12176
Number of cells after MT filter: 10513


In [30]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 10513 × 22976
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [31]:
adata_final.write("data/Mouse_public_data/GSE269449/GSE269449_postQC-R.h5ad")

In [32]:
adata_final.obs['sample'].value_counts()

sample
GSE269449_2    6816
GSE269449_1    3697
Name: count, dtype: int64

## GSE298574 28892 26239

In [33]:
file_paths = [
    "data/Mouse_public_data/GSE298574/GSE298574_KO.h5ad",
    "data/Mouse_public_data/GSE298574/GSE298574_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7766
Number of cells in sample Sample2: 21126


In [34]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE298574/GSE298574_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [35]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [36]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE298574/GSE298574_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE298574/GSE298574_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 17:52:56 2026 .. Analyzing cells in batch 'GSE298574_1'
Thu Apr 23 17:52:56 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 17:53:30 2026 .... Estimating contamination
Thu Apr 23 17:53:37 2026 ...... Completed iteration: 10 | converge: 0.02977
Thu Apr 23 17:53:42 2026 ...... Completed iteration: 20 | converge: 0.01241
Thu Apr 23 17:53:48 2026 ...... Completed iteration: 30 | converge: 0.007056
Thu Apr 23 17:53:53 2026 ...... Completed iteration: 40 | converge: 0.005263
Thu Apr 23 17:53:59 2026 ...... Completed iteration: 50 | converge: 0.004407
Thu Apr 23 17:54:04 2026 ...... Completed iteration: 60 | converge: 0.003613
Thu Apr 23 17:54:09 2026 ...... Completed iteration: 70 | converge: 0.003001
Thu Apr 23 17:54:15 2026 ...... Completed iteration: 80 | converge: 0.002319
Thu Apr 23 17:54:21 2026 ...... Completed iteration: 90 | converge: 0.001

In [37]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE298574/GSE298574_merged_postR.h5ad")

In [38]:
adata_final

AnnData object with n_obs × n_vars = 28892 × 33696
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE298574_1_UMAP', 'decontX_GSE298574_2_UMAP'
    layers: 'decontXcounts'

In [39]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 255 个包含 'mt' 的基因


In [40]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE298574/GSE298574_merged_postR.h5ad")
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [41]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGTATGCAA-1               5862       27665.0       4.370143
AAACCCAAGTTGTAAG-1               1057        2145.0      13.986014
AAACCCACAAAGAGTT-1               2072        5116.0      13.193901
AAACCCACACGGTCTG-1               4961       22302.0       9.339970
AAACCCAGTACCTAAC-1               2185        6663.0       2.416329

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [42]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=8000)
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 30000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 18]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 28892
Number of cells before counts filter: 28255
Number of cells beforeMT filter: 27807
Number of cells after MT filter: 26239


In [43]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 26239 × 24443
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE298574_1_UMAP', 'decontX_GSE298574_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [44]:
adata_final.write("data/Mouse_public_data/GSE298574/GSE298574_postQC-R.h5ad")

In [45]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE298574/GSE298574_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 26239 × 24443
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE298574_1_UMAP', 'decontX_GSE298574_2_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [46]:
adata_final.obs['sample'].value_counts()

sample
GSE298574_2    19413
GSE298574_1     6826
Name: count, dtype: int64

## GSE274572 12258 10515/7525

In [47]:
file_paths = [
    "data/Mouse_public_data/GSE274572/GSE274572_D11.h5ad",
    "data/Mouse_public_data/GSE274572/GSE274572_NS.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 6234
Number of cells in sample Sample2: 6024


In [48]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE274572/GSE274572_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [49]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [50]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE274572/GSE274572_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE274572/GSE274572_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 18:59:06 2026 .. Analyzing cells in batch 'GSE274572_2'
Thu Apr 23 18:59:06 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 18:59:34 2026 .... Estimating contamination
Thu Apr 23 18:59:38 2026 ...... Completed iteration: 10 | converge: 0.02679
Thu Apr 23 18:59:41 2026 ...... Completed iteration: 20 | converge: 0.008858
Thu Apr 23 18:59:45 2026 ...... Completed iteration: 30 | converge: 0.004975
Thu Apr 23 18:59:48 2026 ...... Completed iteration: 40 | converge: 0.002904
Thu Apr 23 18:59:52 2026 ...... Completed iteration: 50 | converge: 0.002092
Thu Apr 23 18:59:56 2026 ...... Completed iteration: 60 | converge: 0.001581
Thu Apr 23 19:00:00 2026 ...... Completed iteration: 70 | converge: 0.001035
Thu Apr 23 19:00:00 2026 ...... Completed iteration: 71 | converge: 0.0009971
Thu Apr 23 19:00:00 2026 .. Calculating final decontaminated matrix
Thu

In [51]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE274572/GSE274572_merged_postR.h5ad")

In [52]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 279 个包含 'mt' 的基因


In [53]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [54]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")
sc.pp.filter_cells(adata_final, min_genes=0) # to get n_genes

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGCGTCGAA-1                740        1236.0       5.987055
AAACCCAAGTCTGGTT-1               4629       20502.0       0.858453
AAACCCACACCTTCGT-1               2345        6596.0       2.152820
AAACCCACATTCACCC-1               1978        4243.0       4.996465
AAACCCAGTGTGATGG-1               1211        2600.0       8.000000

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


In [55]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=2000) 
# sc.pp.filter_cells(adata_final, max_genes=2000)
# sc.pp.filter_genes(adata_final, min_cells=5)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, min_counts = 5000)

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 12258
Number of cells before counts filter: 7933
Number of cells beforeMT filter: 7726
Number of cells after MT filter: 7525


In [56]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 7525 × 55422
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE274572_1_UMAP', 'decontX_GSE274572_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [57]:
adata_final.write("data/Mouse_public_data/GSE274572/GSE274572_postQC-R.h5ad")

In [58]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE274572/GSE274572_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 7525 × 55422
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE274572_1_UMAP', 'decontX_GSE274572_2_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [59]:
adata_final.obs['sample'].value_counts()

sample
GSE274572_1    4043
GSE274572_2    3482
Name: count, dtype: int64

## GSE210406 18545 15843/14231

In [60]:
file_paths = [
    "data/Mouse_public_data/GSE210406/GSE210406_TRF2_HFD.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_TRF2_TA.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_WT_HFD.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 2358
Number of cells in sample Sample2: 6257
Number of cells in sample Sample3: 4250
Number of cells in sample Sample4: 5680


In [61]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE210406/GSE210406_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [62]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [63]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE210406/GSE210406_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE210406/GSE210406_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 19:54:15 2026 .. Analyzing cells in batch 'GSE210406_1'
Thu Apr 23 19:54:15 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 19:54:29 2026 .... Estimating contamination
Thu Apr 23 19:54:32 2026 ...... Completed iteration: 10 | converge: 0.03435
Thu Apr 23 19:54:34 2026 ...... Completed iteration: 20 | converge: 0.01457
Thu Apr 23 19:54:36 2026 ...... Completed iteration: 30 | converge: 0.009001
Thu Apr 23 19:54:38 2026 ...... Completed iteration: 40 | converge: 0.006363
Thu Apr 23 19:54:40 2026 ...... Completed iteration: 50 | converge: 0.002966
Thu Apr 23 19:54:42 2026 ...... Completed iteration: 60 | converge: 0.001841
Thu Apr 23 19:54:44 2026 ...... Completed iteration: 70 | converge: 0.001507
Thu Apr 23 19:54:46 2026 ...... Completed iteration: 80 | converge: 0.001228
Thu Apr 23 19:54:48 2026 ...... Completed iteration: 90 | converge: 0.001

In [64]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE210406/GSE210406_merged_postR.h5ad")

In [65]:
adata_final

AnnData object with n_obs × n_vars = 18545 × 55492
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE210406_1_UMAP', 'decontX_GSE210406_2_UMAP', 'decontX_GSE210406_3_UMAP', 'decontX_GSE210406_4_UMAP'
    layers: 'decontXcounts'

In [66]:
adata_final.var_names

Index(['ENSMUSG00000102693', 'ENSMUSG00000064842', 'ENSMUSG00000051951',
       'ENSMUSG00000102851', 'ENSMUSG00000103377', 'ENSMUSG00000104017',
       'ENSMUSG00000103025', 'ENSMUSG00000089699', 'ENSMUSG00000103201',
       'ENSMUSG00000103147',
       ...
       'ENSMUSG00000094431', 'ENSMUSG00000094621', 'ENSMUSG00000098647',
       'ENSMUSG00000096730', 'ENSMUSG00000095742', 'mCereulean_cDNA',
       'hrGFP_cDNA', 'EYFP_cDNA', 'tdimer2_12_cDNA', 'hTRF2_transgene'],
      dtype='object', length=55492)

In [67]:
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
# 核心修改：创建“基因ID→基因名”的映射字典（原代码是“基因名→基因ID”）
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
# 备份原始的基因ID
adata_final.var['original_ensembl_ids'] = adata_final.var_names

# 将AnnData的变量名（基因ID）转换为基因名
# 逻辑：如果基因ID在映射字典中，则替换为对应的基因名；否则保留原始ID
adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

In [68]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 299 个包含 'mt' 的基因


In [69]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [70]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAGTATTCCGA-1               1256        2823.0       2.798441
AAACCCATCGCCGTGA-1               1373        2439.0       1.517015
AAACGAAGTATTGGCT-1               4152       13688.0       0.694039
AAACGAAGTTAAGGGC-1               1354        2735.0       7.166362
AAACGCTAGGAATTAC-1               2014        4391.0       0.204965

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


In [71]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=1200) 
# sc.pp.filter_cells(adata_final, max_genes=8000)
# sc.pp.filter_genes(adata_final, min_cells=5) 

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
# sc.pp.filter_cells(adata_final, max_counts = 30000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 6]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 18545
Number of cells beforeMT filter: 15346
Number of cells after MT filter: 14231


In [72]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 14231 × 55492
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'original_ensembl_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE210406_1_UMAP', 'decontX_GSE210406_2_UMAP', 'decontX_GSE210406_3_UMAP', 'decontX_GSE210406_4_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [73]:
adata_final.write("data/Mouse_public_data/GSE210406/GSE210406_postQC-R.h5ad")

In [74]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE210406/GSE210406_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 14231 × 55492
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'original_ensembl_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE210406_1_UMAP', 'decontX_GSE210406_2_UMAP', 'decontX_GSE210406_3_UMAP', 'decontX_GSE210406_4_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [75]:
adata_final.obs['sample'].value_counts()

sample
GSE210406_2    5394
GSE210406_3    4966
GSE210406_4    2350
GSE210406_1    1521
Name: count, dtype: int64

## GSE262694 6845804 86091/80165

In [76]:
file_paths = [
    "data/Mouse_public_data/GSE262694/GSE262694_KO_12wks.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_12wks_rep1.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_12wks_rep2.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_30wks_rep1.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_WT_30wks_rep2.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9737
Number of cells in sample Sample2: 9379
Number of cells in sample Sample3: 11720
Number of cells in sample Sample4: 20088
Number of cells in sample Sample5: 6794880


In [77]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE262694/GSE262694_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [78]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE262694/GSE262694_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [79]:
# pp-1
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)
sc.pp.filter_genes(adata_final, min_cells=3) 
print('Number of cells after gene filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 6845804


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells after gene filter: 89101


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [80]:
adata_final.write("data/Mouse_public_data/GSE262694/GSE262694_merged_preR.h5ad")

In [81]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [82]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE262694/GSE262694_merged_preR.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE262694/GSE262694_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 20:30:40 2026 .. Analyzing cells in batch 'GSE262694_1'
Thu Apr 23 20:30:41 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 20:31:16 2026 .... Estimating contamination
Thu Apr 23 20:31:23 2026 ...... Completed iteration: 10 | converge: 0.02732
Thu Apr 23 20:31:28 2026 ...... Completed iteration: 20 | converge: 0.008433
Thu Apr 23 20:31:33 2026 ...... Completed iteration: 30 | converge: 0.004697
Thu Apr 23 20:31:39 2026 ...... Completed iteration: 40 | converge: 0.003003
Thu Apr 23 20:31:44 2026 ...... Completed iteration: 50 | converge: 0.001902
Thu Apr 23 20:31:49 2026 ...... Completed iteration: 60 | converge: 0.001524
Thu Apr 23 20:31:55 2026 ...... Completed iteration: 70 | converge: 0.00118
Thu Apr 23 20:31:57 2026 ...... Completed iteration: 74 | converge: 0.0009564
Thu Apr 23 20:31:57 2026 .. Calculating final decontaminated matrix
Thu 

In [83]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE262694/GSE262694_merged_postR.h5ad")

In [84]:
adata_final

AnnData object with n_obs × n_vars = 89101 × 20333
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    var: 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE262694_1_UMAP', 'decontX_GSE262694_2_UMAP', 'decontX_GSE262694_3_UMAP', 'decontX_GSE262694_4_UMAP', 'decontX_GSE262694_5_UMAP'
    layers: 'decontXcounts'

In [85]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 220 个包含 'mt' 的基因


In [86]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [87]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACAACTCCAA-1               5289       21333.0       3.698495
AAACCCACACCTGCGA-1               3188       11756.0       2.152093
AAACCCACACGCAAAG-1               2924       15642.0       1.726122
AAACCCACACGCGTCA-1               4132       16640.0       3.804087
AAACCCACAGGACATG-1               1902        5635.0       3.229814

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [88]:
adata_final.obs_names_make_unique()
adata_final.var_names_make_unique()

In [89]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
# sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=3) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 20000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 89101
Number of cells before counts filter: 89101
Number of cells beforeMT filter: 83174
Number of cells after MT filter: 80165


In [90]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 80165 × 20333
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_counts'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE262694_1_UMAP', 'decontX_GSE262694_2_UMAP', 'decontX_GSE262694_3_UMAP', 'decontX_GSE262694_4_UMAP', 'decontX_GSE262694_5_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [91]:
adata_final.write("data/Mouse_public_data/GSE262694/GSE262694_postQC-R.h5ad")

In [92]:
adata_final.obs['sample'].value_counts()

sample
GSE262694_5    37572
GSE262694_4    18449
GSE262694_3     9386
GSE262694_1     7542
GSE262694_2     7216
Name: count, dtype: int64

## GSE284253 5475 4961

In [93]:
file_paths = [
    "data/Mouse_public_data/GSE284253/GSE284253_AAA.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_Naive.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_RA.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 2842
Number of cells in sample Sample2: 782
Number of cells in sample Sample3: 1851


In [94]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE284253/GSE284253_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [95]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [96]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE284253/GSE284253_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE284253/GSE284253_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:02:04 2026 .. Analyzing cells in batch 'GSE284253_1'
Thu Apr 23 21:02:04 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:02:27 2026 .... Estimating contamination
Thu Apr 23 21:02:28 2026 ...... Completed iteration: 10 | converge: 0.02637
Thu Apr 23 21:02:29 2026 ...... Completed iteration: 20 | converge: 0.007659
Thu Apr 23 21:02:31 2026 ...... Completed iteration: 30 | converge: 0.004208
Thu Apr 23 21:02:32 2026 ...... Completed iteration: 40 | converge: 0.002719
Thu Apr 23 21:02:33 2026 ...... Completed iteration: 50 | converge: 0.00178
Thu Apr 23 21:02:35 2026 ...... Completed iteration: 60 | converge: 0.001372
Thu Apr 23 21:02:36 2026 ...... Completed iteration: 70 | converge: 0.001185
Thu Apr 23 21:02:37 2026 ...... Completed iteration: 78 | converge: 0.0009735
Thu Apr 23 21:02:37 2026 .. Calculating final decontaminated matrix
Thu 

In [97]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE284253/GSE284253_merged_postR.h5ad")

In [98]:
adata_final

AnnData object with n_obs × n_vars = 5475 × 27998
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP'
    layers: 'decontXcounts'

In [99]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [100]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [101]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCGATAGC-1               3471       13631.0       2.376935
AAACCTGAGTGGCACA-1               3147       12301.0       1.796602
AAACCTGCATTAACCG-1               1382        4020.0       3.482587
AAACCTGGTCCAGTGC-1               1446        3774.0       1.854796
AAACCTGGTGGTCCGT-1                892        2073.0       4.823927

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [102]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300) 
sc.pp.filter_cells(adata_final, max_genes=3000) 
sc.pp.filter_genes(adata_final, min_cells=5)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 7.5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5475
Number of cells before counts filter: 4981
Number of cells beforeMT filter: 4981
Number of cells after MT filter: 4961


In [103]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 4961 × 14092
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [104]:
adata_final.write("data/Mouse_public_data/GSE284253/GSE284253_postQC-R.h5ad")

In [105]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE284253/GSE284253_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 4961 × 14092
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [106]:
adata_final.obs['sample'].value_counts()

sample
GSE284253_1    2533
GSE284253_3    1656
GSE284253_2     772
Name: count, dtype: int64

## GSE279601 2182871 15679/15836

In [107]:
file_paths = [
    "data/Mouse_public_data/GSE279601/GSE279601_rep1_MT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep1_WT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep2_MT.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_rep2_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 487632
Number of cells in sample Sample2: 509697
Number of cells in sample Sample3: 610398
Number of cells in sample Sample4: 575144


In [108]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [109]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged.h5ad")
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
print('Number of cells afetr gene filter: {:d}'.format(adata_final.n_obs))

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of cells before gene filter: 2182871
Number of cells afetr gene filter: 16453


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [110]:
adata_final.write_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged_preR.h5ad")

In [111]:
adata_final

AnnData object with n_obs × n_vars = 16453 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes'

In [112]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [113]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE279601/GSE279601_merged_preR.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE279601/GSE279601_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:08:21 2026 .. Analyzing cells in batch 'GSE279601_1'
Thu Apr 23 21:08:21 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:08:47 2026 .... Estimating contamination
Thu Apr 23 21:08:50 2026 ...... Completed iteration: 10 | converge: 0.02558
Thu Apr 23 21:08:52 2026 ...... Completed iteration: 20 | converge: 0.008757
Thu Apr 23 21:08:54 2026 ...... Completed iteration: 30 | converge: 0.004811
Thu Apr 23 21:08:57 2026 ...... Completed iteration: 40 | converge: 0.003251
Thu Apr 23 21:08:59 2026 ...... Completed iteration: 50 | converge: 0.002299
Thu Apr 23 21:09:02 2026 ...... Completed iteration: 60 | converge: 0.001712
Thu Apr 23 21:09:04 2026 ...... Completed iteration: 70 | converge: 0.001302
Thu Apr 23 21:09:06 2026 ...... Completed iteration: 80 | converge: 0.001001
Thu Apr 23 21:09:07 2026 ...... Completed iteration: 81 | converge: 0.00

In [114]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE279601/GSE279601_merged_postR.h5ad")

In [115]:
adata_final

AnnData object with n_obs × n_vars = 16453 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP'
    layers: 'decontXcounts'

In [116]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [117]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [118]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGTCCTACA-1               2300        6014.0       1.845693
AAACCCACAGAATTCC-1               1679        5418.0       0.258398
AAACCCAGTGATGGCA-1               2022        5308.0       1.281085
AAACCCAGTTAGTCGT-1                650        1176.0      10.459184
AAACGAAAGAGGATCC-1                445         764.0      10.994764

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [119]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_genes(adata_final, min_cells=3) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 16453
Number of cells beforeMT filter: 16453
Number of cells after MT filter: 15836


In [120]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 15836 × 17576
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'n_genes', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [121]:
adata_final.write("data/Mouse_public_data/GSE279601/GSE279601_postQC-R.h5ad")

In [122]:
adata_final.obs['sample'].value_counts()

sample
GSE279601_1    5567
GSE279601_2    4036
GSE279601_3    3629
GSE279601_4    2604
Name: count, dtype: int64

## GSE211216 7673 6857/6124

In [123]:
file_paths = [
    "data/Mouse_public_data/GSE211216/GSE211216_Control.h5ad",
    "data/Mouse_public_data/GSE211216/GSE211216_Diabetes.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    adata_sample.var_names_make_unique()
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample1: 4215
Number of cells in sample Sample2: 3458


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [124]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE211216/GSE211216_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [125]:
adata_final

AnnData object with n_obs × n_vars = 7673 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location'

In [126]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [127]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE211216/GSE211216_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE211216/GSE211216_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:15:51 2026 .. Analyzing cells in batch 'GSE211216_1'
Thu Apr 23 21:15:51 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:16:15 2026 .... Estimating contamination
Thu Apr 23 21:16:17 2026 ...... Completed iteration: 10 | converge: 0.02492
Thu Apr 23 21:16:19 2026 ...... Completed iteration: 20 | converge: 0.01104
Thu Apr 23 21:16:20 2026 ...... Completed iteration: 30 | converge: 0.006808
Thu Apr 23 21:16:22 2026 ...... Completed iteration: 40 | converge: 0.004677
Thu Apr 23 21:16:24 2026 ...... Completed iteration: 50 | converge: 0.003463
Thu Apr 23 21:16:26 2026 ...... Completed iteration: 60 | converge: 0.0026
Thu Apr 23 21:16:28 2026 ...... Completed iteration: 70 | converge: 0.001991
Thu Apr 23 21:16:30 2026 ...... Completed iteration: 80 | converge: 0.001648
Thu Apr 23 21:16:32 2026 ...... Completed iteration: 90 | converge: 0.00140

In [128]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE211216/GSE211216_merged_postR.h5ad")

In [129]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [130]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [131]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGAGACTTA-1               1170        2175.0       4.183908
AAACCTGAGAGTGAGA-1               2019        7244.0       1.697957
AAACCTGAGATCCGAG-1               1367        3298.0       1.425106
AAACCTGAGCTTATCG-1               1858        5629.0       4.707763
AAACCTGAGTCACGCC-1                787        1603.0       2.557704

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [132]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300) 
sc.pp.filter_cells(adata_final, max_genes=3000)
print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 1000)
sc.pp.filter_cells(adata_final, max_counts = 23000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 7673
Number of cells before counts filter: 6726
Number of cells beforeMT filter: 6281
Number of cells after MT filter: 6124


In [133]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 6124 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE211216_1_UMAP', 'decontX_GSE211216_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [134]:
adata_final.write("data/Mouse_public_data/GSE211216/GSE211216_postQC-R.h5ad")

In [135]:
adata_final.obs['sample'].value_counts()

sample
GSE211216_1    3768
GSE211216_2    2356
Name: count, dtype: int64

## GSE245373 5979 5821

In [136]:
file_paths = [
    "data/Mouse_public_data/GSE245373/GSE245373_advAp.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_advWT.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_bAPOE.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3153
Number of cells in sample Sample2: 2271
Number of cells in sample Sample3: 555


In [137]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE245373/GSE245373_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [138]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [139]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE245373/GSE245373_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE245373/GSE245373_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:20:25 2026 .. Analyzing cells in batch 'GSE245373_1'
Thu Apr 23 21:20:25 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:20:45 2026 .... Estimating contamination
Thu Apr 23 21:20:46 2026 ...... Completed iteration: 10 | converge: 0.02074
Thu Apr 23 21:20:48 2026 ...... Completed iteration: 20 | converge: 0.005514
Thu Apr 23 21:20:49 2026 ...... Completed iteration: 30 | converge: 0.00268
Thu Apr 23 21:20:50 2026 ...... Completed iteration: 40 | converge: 0.001458
Thu Apr 23 21:20:51 2026 ...... Completed iteration: 50 | converge: 0.001443
Thu Apr 23 21:20:52 2026 ...... Completed iteration: 60 | converge: 0.001369
Thu Apr 23 21:20:53 2026 ...... Completed iteration: 65 | converge: 0.0009384
Thu Apr 23 21:20:53 2026 .. Calculating final decontaminated matrix
Thu Apr 23 21:20:54 2026 .. Analyzing cells in batch 'GSE245373_2'
Thu Apr 23 21:

In [140]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE245373/GSE245373_merged_postR.h5ad")

In [141]:
adata_final

AnnData object with n_obs × n_vars = 5979 × 55358
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE245373_1_UMAP', 'decontX_GSE245373_2_UMAP', 'decontX_GSE245373_3_UMAP'
    layers: 'decontXcounts'

In [142]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 13 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 462 个包含 'mt' 的基因


In [143]:
mt_genes_upper

['MT-ATP6',
 'MT-ATP8',
 'MT-CO1',
 'MT-CO2',
 'MT-CO3',
 'MT-CYTB',
 'MT-ND1',
 'MT-ND2',
 'MT-ND3',
 'MT-ND4',
 'MT-ND4L',
 'MT-ND5',
 'MT-ND6']

In [144]:
mt_genes_lower

['mt-Atp6',
 'mt-Atp8',
 'mt-Co1',
 'mt-Co2',
 'mt-Co3',
 'mt-Cytb',
 'mt-Nd1',
 'mt-Nd2',
 'mt-Nd3',
 'mt-Nd4',
 'mt-Nd4l',
 'mt-Nd5',
 'mt-Nd6']

In [145]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [146]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCTAACTC-1                725        2256.0            0.0
AAACCTGCAATTCCTT-1               2473        7937.0            0.0
AAACCTGCACACCGAC-1               1451        9171.0            0.0
AAACCTGCATAGGATA-1                861        1744.0            0.0
AAACCTGCATCAGTCA-1                898        2253.0            0.0

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Atp6', 'mt-Atp8', 'mt-Co1', 'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4', 'mt-Nd4l', 'mt-Nd5', 'mt-Nd6']


In [147]:
print(adata_final.obs['pct_counts_mt'].value_counts())

pct_counts_mt
0.000000    3708
1.785714       3
1.550388       3
1.666667       2
1.574803       2
            ... 
1.403785       1
1.311189       1
0.732104       1
1.522567       1
1.057318       1
Name: count, Length: 2239, dtype: int64


In [148]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=5000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 200000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 3] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5979
Number of cells before counts filter: 5972
Number of cells beforeMT filter: 5972
Number of cells after MT filter: 5821


In [149]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 5821 × 34297
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE245373_1_UMAP', 'decontX_GSE245373_2_UMAP', 'decontX_GSE245373_3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [150]:
adata_final.write("data/Mouse_public_data/GSE245373/GSE245373_postQC_mt-R.h5ad")

In [151]:
adata_final.obs['sample'].value_counts()

sample
GSE245373_1    3150
GSE245373_2    2117
GSE245373_3     554
Name: count, dtype: int64

## GSE264253 11102 9915/7156

In [152]:
file_paths = [
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_CD31.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_CD45.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_Live.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_Ab_YFP.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_CD31.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_CD45.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_Live.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_IgG_YFP.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 295
Number of cells in sample Sample2: 1621
Number of cells in sample Sample3: 505
Number of cells in sample Sample4: 5789
Number of cells in sample Sample5: 80
Number of cells in sample Sample6: 560
Number of cells in sample Sample7: 404
Number of cells in sample Sample8: 1848


In [153]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE264253/GSE264253_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [154]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [155]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE264253/GSE264253_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE264253/GSE264253_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:34:15 2026 .. Analyzing cells in batch 'GSE264253_1'
Thu Apr 23 21:34:15 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:34:21 2026 .... Estimating contamination
Thu Apr 23 21:34:22 2026 ...... Completed iteration: 10 | converge: 0.005657
Thu Apr 23 21:34:22 2026 ...... Completed iteration: 20 | converge: 0.001509
Thu Apr 23 21:34:22 2026 ...... Completed iteration: 22 | converge: 0.0009709
Thu Apr 23 21:34:22 2026 .. Calculating final decontaminated matrix
Thu Apr 23 21:34:22 2026 .. Analyzing cells in batch 'GSE264253_2'
Thu Apr 23 21:34:22 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:34:32 2026 .... Estimating contamination
Thu Apr 23 21:34:33 2026 ...... Completed iteration: 10 | converge: 0.02624
Thu Apr 23 21:34:34 2026 ...... Completed iteration: 20 | converge: 0.008427
Thu Apr 23 21:34:35 2026 ...... Complete

In [156]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE264253/GSE264253_merged_postR.h5ad")

In [157]:
adata_final

AnnData object with n_obs × n_vars = 11102 × 28693
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE264253_1_UMAP', 'decontX_GSE264253_2_UMAP', 'decontX_GSE264253_3_UMAP', 'decontX_GSE264253_4_UMAP', 'decontX_GSE264253_5_UMAP', 'decontX_GSE264253_6_UMAP', 'decontX_GSE264253_7_UMAP', 'decontX_GSE264253_8_UMAP'
    layers: 'decontXcounts'

In [158]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [159]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [160]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACGGGGTTACTGAC-1                237         562.0      37.366548
AAACGGGTCCACTCCA-1               2420        7621.0       2.256922
AAAGCAAAGGGCATGT-1               4639       21825.0       0.902635
AAAGTAGAGAGGTACC-1               4919       32268.0       1.050576
AAATGCCTCCGTTGTC-1               3298       11593.0       0.810834

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [161]:
# pp
# print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
# sc.pp.filter_cells(adata_final, min_genes=200) 
# sc.pp.filter_cells(adata_final, max_genes=5000) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 200)
sc.pp.filter_cells(adata_final, max_counts = 5000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before counts filter: 11102
Number of cells beforeMT filter: 7482
Number of cells after MT filter: 7156


In [162]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 7156 × 28693
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE264253_1_UMAP', 'decontX_GSE264253_2_UMAP', 'decontX_GSE264253_3_UMAP', 'decontX_GSE264253_4_UMAP', 'decontX_GSE264253_5_UMAP', 'decontX_GSE264253_6_UMAP', 'decontX_GSE264253_7_UMAP', 'decontX_GSE264253_8_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [163]:
adata_final.write("data/Mouse_public_data/GSE264253/GSE264253_postQC-R.h5ad")

In [164]:
adata_final.obs['sample'].value_counts()

sample
GSE264253_4    4637
GSE264253_2     893
GSE264253_8     518
GSE264253_6     462
GSE264253_7     294
GSE264253_3     268
GSE264253_1      49
GSE264253_5      35
Name: count, dtype: int64

## GSE209525 16820 15387

In [165]:
file_paths = [
    "data/Mouse_public_data/GSE209525/GSE209525_1_ND.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_2_ND.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_3_HFD.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_4_HFD.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_5_HFD_VG.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_6_HFD_VG.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3399
Number of cells in sample Sample2: 1577
Number of cells in sample Sample3: 4055
Number of cells in sample Sample4: 1449
Number of cells in sample Sample5: 2725
Number of cells in sample Sample6: 3615


In [166]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE209525/GSE209525_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [167]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [168]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE209525/GSE209525_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE209525/GSE209525_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:39:55 2026 .. Analyzing cells in batch 'GSE209525_1'
Thu Apr 23 21:39:55 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:40:11 2026 .... Estimating contamination
Thu Apr 23 21:40:13 2026 ...... Completed iteration: 10 | converge: 0.03007
Thu Apr 23 21:40:14 2026 ...... Completed iteration: 20 | converge: 0.01362
Thu Apr 23 21:40:15 2026 ...... Completed iteration: 30 | converge: 0.005391
Thu Apr 23 21:40:17 2026 ...... Completed iteration: 40 | converge: 0.003772
Thu Apr 23 21:40:18 2026 ...... Completed iteration: 50 | converge: 0.002931
Thu Apr 23 21:40:19 2026 ...... Completed iteration: 60 | converge: 0.002459
Thu Apr 23 21:40:21 2026 ...... Completed iteration: 70 | converge: 0.001773
Thu Apr 23 21:40:22 2026 ...... Completed iteration: 80 | converge: 0.001906
Thu Apr 23 21:40:24 2026 ...... Completed iteration: 90 | converge: 0.001

In [169]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE209525/GSE209525_merged_postR.h5ad")

In [170]:
adata_final

AnnData object with n_obs × n_vars = 16820 × 32288
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE209525_1_UMAP', 'decontX_GSE209525_2_UMAP', 'decontX_GSE209525_3_UMAP', 'decontX_GSE209525_4_UMAP', 'decontX_GSE209525_5_UMAP', 'decontX_GSE209525_6_UMAP'
    layers: 'decontXcounts'

In [171]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [172]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [173]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGGTAGATT-1               1262        2316.0       1.424870
AAACCCACATAAGCAA-1               2221        7483.0       2.138180
AAACCCACATACATCG-1               4553       17310.0       2.403235
AAACCCAGTGGTCTTA-1               2200        4852.0       1.957955
AAACCCATCAAATGCC-1               4910       25370.0       5.514387

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [174]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  #acc to author
sc.pp.filter_cells(adata_final, max_genes=10000) # acc to author
sc.pp.filter_genes(adata_final, min_cells=1) #acc to author

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 30000) #acc to author


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] #acc to author
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 16820
Number of cells before counts filter: 16673
Number of cells beforeMT filter: 16474
Number of cells after MT filter: 15387


In [175]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 15387 × 25619
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE209525_1_UMAP', 'decontX_GSE209525_2_UMAP', 'decontX_GSE209525_3_UMAP', 'decontX_GSE209525_4_UMAP', 'decontX_GSE209525_5_UMAP', 'decontX_GSE209525_6_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [176]:
adata_final.write("data/Mouse_public_data/GSE209525/GSE209525_postQC-R.h5ad")

In [177]:
adata_final.obs['sample'].value_counts()

sample
GSE209525_3    3483
GSE209525_6    3439
GSE209525_1    3246
GSE209525_5    2479
GSE209525_2    1475
GSE209525_4    1265
Name: count, dtype: int64

## GSE205930 40208 38460

In [178]:
file_paths = [
    "data/Mouse_public_data/GSE205930/GSE205930_C_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_ED_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_IC_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_LD_AO.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_PL_AO.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 10704
Number of cells in sample Sample2: 7019
Number of cells in sample Sample3: 7756
Number of cells in sample Sample4: 9030
Number of cells in sample Sample5: 5699


In [179]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE205930/GSE205930_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [180]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [181]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE205930/GSE205930_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE205930/GSE205930_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 21:51:21 2026 .. Analyzing cells in batch 'GSE205930_1'
Thu Apr 23 21:51:21 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 21:51:57 2026 .... Estimating contamination
Thu Apr 23 21:52:02 2026 ...... Completed iteration: 10 | converge: 0.0206
Thu Apr 23 21:52:05 2026 ...... Completed iteration: 20 | converge: 0.01002
Thu Apr 23 21:52:09 2026 ...... Completed iteration: 30 | converge: 0.007678
Thu Apr 23 21:52:13 2026 ...... Completed iteration: 40 | converge: 0.006001
Thu Apr 23 21:52:17 2026 ...... Completed iteration: 50 | converge: 0.004806
Thu Apr 23 21:52:22 2026 ...... Completed iteration: 60 | converge: 0.003885
Thu Apr 23 21:52:27 2026 ...... Completed iteration: 70 | converge: 0.003404
Thu Apr 23 21:52:32 2026 ...... Completed iteration: 80 | converge: 0.002534
Thu Apr 23 21:52:35 2026 ...... Completed iteration: 90 | converge: 0.0023

In [182]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE205930/GSE205930_merged_postR.h5ad")

In [183]:
adata_final

AnnData object with n_obs × n_vars = 40208 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE205930_1_UMAP', 'decontX_GSE205930_2_UMAP', 'decontX_GSE205930_3_UMAP', 'decontX_GSE205930_4_UMAP', 'decontX_GSE205930_5_UMAP'
    layers: 'decontXcounts'

In [184]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [185]:

adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [186]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGCAGATCG-1               1045        2167.0       2.768805
AAACCTGAGCGATTCT-1               1006        2182.0      11.274060
AAACCTGAGTTGAGTA-1               1886        4501.0       6.420795
AAACCTGCAAGCCCAC-1               1581        3441.0       8.137169
AAACCTGCACAGACTT-1               1922        4640.0       7.650862

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [187]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=100)  
sc.pp.filter_cells(adata_final, max_genes=10000) 
sc.pp.filter_genes(adata_final, min_cells=1)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 10000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 40208
Number of cells before counts filter: 40092
Number of cells beforeMT filter: 39793
Number of cells after MT filter: 38460


In [188]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 38460 × 20260
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE205930_1_UMAP', 'decontX_GSE205930_2_UMAP', 'decontX_GSE205930_3_UMAP', 'decontX_GSE205930_4_UMAP', 'decontX_GSE205930_5_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [189]:
adata_final.write("data/Mouse_public_data/GSE205930/GSE205930_postQC-R.h5ad")

In [190]:
adata_final.obs['sample'].value_counts()

sample
GSE205930_1    10214
GSE205930_4     8228
GSE205930_3     7469
GSE205930_2     7001
GSE205930_5     5548
Name: count, dtype: int64

## GSE155513 42372 40958/42372

In [191]:
file_paths = [
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_ApoE_KO_22_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_0_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen-_Ldlr_KO_26_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_ApoE_KO_22_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_0_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_8_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_16_wks_WD.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_ZsGreen+_Ldlr_KO_26_wks_WD.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9",
    "Sample10",
    "Sample11",
    "Sample12",
    "Sample13",
    "Sample14"
    
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 1935
Number of cells in sample Sample2: 1575
Number of cells in sample Sample3: 2062
Number of cells in sample Sample4: 2498
Number of cells in sample Sample5: 1714
Number of cells in sample Sample6: 2742
Number of cells in sample Sample7: 3403
Number of cells in sample Sample8: 2932
Number of cells in sample Sample9: 3150
Number of cells in sample Sample10: 3426
Number of cells in sample Sample11: 3094
Number of cells in sample Sample12: 3513
Number of cells in sample Sample13: 4287
Number of cells in sample Sample14: 6041


In [192]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/GSE155513/GSE155513_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [193]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [194]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE155513/GSE155513_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE155513/GSE155513_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Thu Apr 23 22:12:33 2026 .. Analyzing cells in batch 'GSE155513_10'
Thu Apr 23 22:12:33 2026 .... Generating UMAP and estimating cell types
Thu Apr 23 22:12:43 2026 .... Estimating contamination
Thu Apr 23 22:12:44 2026 ...... Completed iteration: 10 | converge: 0.02205
Thu Apr 23 22:12:45 2026 ...... Completed iteration: 20 | converge: 0.008611
Thu Apr 23 22:12:46 2026 ...... Completed iteration: 30 | converge: 0.004629
Thu Apr 23 22:12:47 2026 ...... Completed iteration: 40 | converge: 0.002772
Thu Apr 23 22:12:47 2026 ...... Completed iteration: 50 | converge: 0.001805
Thu Apr 23 22:12:48 2026 ...... Completed iteration: 60 | converge: 0.001132
Thu Apr 23 22:12:49 2026 ...... Completed iteration: 63 | converge: 0.0009766
Thu Apr 23 22:12:49 2026 .. Calculating final decontaminated matrix
Thu Apr 23 22:12:49 2026 .. Analyzing cells in batch 'GSE155513_12'
Thu Apr 23 

In [195]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE155513/GSE155513_merged_postR.h5ad")

In [196]:
adata_final

AnnData object with n_obs × n_vars = 42372 × 15852
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE155513_10_UMAP', 'decontX_GSE155513_11_UMAP', 'decontX_GSE155513_12_UMAP', 'decontX_GSE155513_13_UMAP', 'decontX_GSE155513_14_UMAP', 'decontX_GSE155513_1_UMAP', 'decontX_GSE155513_2_UMAP', 'decontX_GSE155513_3_UMAP', 'decontX_GSE155513_4_UMAP', 'decontX_GSE155513_5_UMAP', 'decontX_GSE155513_6_UMAP', 'decontX_GSE155513_7_UMAP', 'decontX_GSE155513_8_UMAP', 'decontX_GSE155513_9_UMAP'
    layers: 'decontXcounts'

In [197]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 207 个包含 'mt' 的基因


In [198]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [199]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                  n_genes_by_counts  total_counts  pct_counts_mt
TCTGAGAGTAAGTAGT               1280        2908.0       0.309491
TGATTTCAGCGTTTAC               1985        5160.0       1.279070
AGCGTCGAGATGGCGT               1441        3313.0       0.724419
GATTCAGCAACTGCTA               1463        3296.0       0.940534
CTCTAATCACATCTTT               1624        3802.0       0.789058

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Atp6', 'mt-Atp8', 'mt-Co1', 'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4', 'mt-Nd4l', 'mt-Nd5', 'mt-Nd6']


In [200]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
# sc.pp.filter_cells(adata_final, max_genes=5000)
# sc.pp.filter_genes(adata_final, min_cells=5)

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

# sc.pp.filter_cells(adata_final, max_counts = 15000) 

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 42372
Number of cells beforeMT filter: 42372
Number of cells after MT filter: 42372


In [201]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 42372 × 15852
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE155513_10_UMAP', 'decontX_GSE155513_11_UMAP', 'decontX_GSE155513_12_UMAP', 'decontX_GSE155513_13_UMAP', 'decontX_GSE155513_14_UMAP', 'decontX_GSE155513_1_UMAP', 'decontX_GSE155513_2_UMAP', 'decontX_GSE155513_3_UMAP', 'decontX_GSE155513_4_UMAP', 'decontX_GSE155513_5_UMAP', 'decontX_GSE155513_6_UMAP', 'decontX_GSE155513_7_UMAP', 'decontX_GSE155513_8_UMAP', 'decontX_GSE155513_9_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [202]:
adata_final.write("data/Mouse_public_data/GSE155513/GSE155513_postQC-R.h5ad")

In [203]:
adata_final.obs['sample'].value_counts()

sample
GSE155513_7     6041
GSE155513_5     4287
GSE155513_3     3513
GSE155513_13    3426
GSE155513_8     3403
GSE155513_11    3150
GSE155513_1     3094
GSE155513_9     2932
GSE155513_6     2742
GSE155513_2     2498
GSE155513_14    2062
GSE155513_10    1935
GSE155513_4     1714
GSE155513_12    1575
Name: count, dtype: int64

## GSE134557 2547 2160/2439

In [204]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE134557/GSE134557.h5ad")

In [205]:
adata_final.obs['sample'].value_counts()

sample
apoend     851
apoehfd    762
dkohfd     668
dkond      266
Name: count, dtype: int64

In [206]:
adata_final

AnnData object with n_obs × n_vars = 2547 × 11326
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention'
    var: 'gene_id'

In [207]:
adata_final.var_names

Index(['Gnai3', 'Narf', 'Cav2', 'Klf6', 'Scmh1', 'Cox5a', 'Wnt9a', 'Fer',
       'Xpo6', 'Tfe3',
       ...
       'CU074400.1', 'AC170188.1', 'AC123738.1', 'AC154176.2', 'AC123935.1',
       'AC154767.3', 'AC154335.1', 'CT025566.1', 'AC126549.2', 'Nupl1'],
      dtype='object', name='gene_name', length=11326)

In [208]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [209]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE134557/GSE134557.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE134557/GSE134557_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 10:19:51 2026 .. Analyzing cells in batch 'apoend'
Fri Apr 24 10:19:51 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 10:19:59 2026 .... Estimating contamination
Fri Apr 24 10:19:59 2026 ...... Completed iteration: 10 | converge: 0.01973
Fri Apr 24 10:20:00 2026 ...... Completed iteration: 20 | converge: 0.008872
Fri Apr 24 10:20:00 2026 ...... Completed iteration: 30 | converge: 0.005639
Fri Apr 24 10:20:00 2026 ...... Completed iteration: 40 | converge: 0.004171
Fri Apr 24 10:20:01 2026 ...... Completed iteration: 50 | converge: 0.003012
Fri Apr 24 10:20:01 2026 ...... Completed iteration: 60 | converge: 0.002475
Fri Apr 24 10:20:02 2026 ...... Completed iteration: 70 | converge: 0.002118
Fri Apr 24 10:20:02 2026 ...... Completed iteration: 80 | converge: 0.001823
Fri Apr 24 10:20:02 2026 ...... Completed iteration: 90 | converge: 0.001396


In [210]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE134557/GSE134557_postR.h5ad")

In [211]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 0 个以 'mt-' 开头的基因
找到 165 个包含 'mt' 的基因


In [212]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [213]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                         n_genes_by_counts  total_counts  pct_counts_mt
AAACGGGGTCAGATAA_apoend               3198   4619.825156            0.0
AAACGGGGTGTGACCC_apoend               3103   4687.382232            0.0
AAAGATGAGATATGCA_apoend               1554   3055.609811            0.0
AAAGATGAGCGTTTAC_apoend               3007   4737.938297            0.0
AAAGATGCATCCGCGA_apoend               1537   3045.752125            0.0

线粒体基因统计：
共标记了 0 个线粒体基因
这些基因是：[]


In [214]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=1500) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 3000) 
print('Number of cells after counts filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 2547
Number of cells before counts filter: 2537
Number of cells after counts filter: 2439


In [215]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

AnnData object with n_obs × n_vars = 2439 × 11326
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_apoehfd_UMAP', 'decontX_apoend_UMAP', 'decontX_dkohfd_UMAP', 'decontX_dkond_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [216]:
adata_final.write("data/Mouse_public_data/GSE134557/GSE134557_postQC-R.h5ad")

In [217]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE134557/GSE134557_postQC-R.h5ad")

In [218]:
adata_final.obs['sample'].value_counts()

sample
apoend     818
apoehfd    734
dkohfd     634
dkond      253
Name: count, dtype: int64

In [219]:
sample_mapping = {
    'apoend': 'GSE134557_1',
    'apoehfd': 'GSE134557_2',
    'dkohfd': 'GSE134557_4',
    'dkond': 'GSE134557_3'
}
adata_final.obs['sample'] = adata_final.obs['sample'].astype(str)
adata_final.obs['sample'] = adata_final.obs['sample'].map(sample_mapping)

In [220]:
adata_final.obs['sample'].value_counts()

sample
GSE134557_1    818
GSE134557_2    734
GSE134557_4    634
GSE134557_3    253
Name: count, dtype: int64

In [221]:
adata_final.write("data/Mouse_public_data/GSE134557/GSE134557_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 2439 × 11326
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_apoehfd_UMAP', 'decontX_apoend_UMAP', 'decontX_dkohfd_UMAP', 'decontX_dkond_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

## GSE131776 27086 26203/27075

In [222]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE131776/GSE131776.h5ad")

In [223]:
adata_final.obs['sample'].value_counts()

sample
GSE131776_3     2436
GSE131776_4     2371
GSE131776_16    2124
GSE131776_9     2091
GSE131776_8     2079
GSE131776_7     2008
GSE131776_6     1856
GSE131776_17    1707
GSE131776_5     1383
GSE131776_12    1371
GSE131776_11    1369
GSE131776_18    1351
GSE131776_1     1271
GSE131776_15    1125
GSE131776_2      758
GSE131776_13     667
GSE131776_10     590
GSE131776_14     529
Name: count, dtype: int64

In [224]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [225]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/GSE131776/GSE131776.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/GSE131776/GSE131776_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 10:47:25 2026 .. Analyzing cells in batch 'GSE131776_1'
Fri Apr 24 10:47:25 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 10:47:34 2026 .... Estimating contamination
Fri Apr 24 10:47:34 2026 ...... Completed iteration: 10 | converge: 0.01368
Fri Apr 24 10:47:35 2026 ...... Completed iteration: 20 | converge: 0.005799
Fri Apr 24 10:47:35 2026 ...... Completed iteration: 30 | converge: 0.003413
Fri Apr 24 10:47:36 2026 ...... Completed iteration: 40 | converge: 0.00227
Fri Apr 24 10:47:37 2026 ...... Completed iteration: 50 | converge: 0.001825
Fri Apr 24 10:47:37 2026 ...... Completed iteration: 60 | converge: 0.001131
Fri Apr 24 10:47:37 2026 ...... Completed iteration: 63 | converge: 0.0009867
Fri Apr 24 10:47:37 2026 .. Calculating final decontaminated matrix
Fri Apr 24 10:47:38 2026 .. Analyzing cells in batch 'GSE131776_2'
Fri Apr 24 10:

In [226]:
adata_final = sc.read_h5ad("data/Mouse_public_data/GSE131776/GSE131776_postR.h5ad")

In [227]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 208 个包含 'mt' 的基因


In [228]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [229]:
adata_final.var['mt']

Xkr4       False
Rp1        False
Sox17      False
Gm37323    False
Mrpl15     False
           ...  
mt-Nd4l     True
mt-Nd4      True
mt-Nd5      True
mt-Nd6      True
mt-Cytb     True
Name: mt, Length: 17976, dtype: bool

In [230]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)
sc.pp.filter_cells(adata_final, max_genes=3500) 
sc.pp.filter_genes(adata_final, min_cells=5) 

# print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

# sc.pp.filter_cells(adata_final, max_counts = 20000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 7.5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))


Number of cells before gene filter: 27086


Number of cells beforeMT filter: 27086
Number of cells after MT filter: 27075


In [231]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 27075 × 17777
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE131776_1_UMAP', 'decontX_GSE131776_2_UMAP', 'decontX_GSE131776_3_UMAP', 'decontX_GSE131776_4_UMAP', 'decontX_GSE131776_5_UMAP', 'decontX_GSE131776_6_UMAP', 'decontX_GSE131776_7_UMAP

In [232]:
adata_final.write("data/Mouse_public_data/GSE131776/GSE131776_postQC-R.h5ad")

In [233]:
adata_final

AnnData object with n_obs × n_vars = 27075 × 17777
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'location', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE131776_10_UMAP', 'decontX_GSE131776_11_UMAP', 'decontX_GSE131776_12_UMAP', 'decontX_GSE131776_13_UMAP', 'decontX_GSE131776_14_UMAP', 'decontX_GSE131776_15_UMAP', 'decontX_GSE131776_16_UMAP', 'decontX_GSE131776_17_UMAP', 'decontX_GSE131776_18_UMAP', 'decontX_GSE131776_1_UMAP', 'decontX_GSE131776_2_UMAP', 'decontX_GSE131776_3_UMAP', 'decontX_GSE131776_4_UMAP', 'decontX_GSE131776_5_UMAP', 'decontX_GSE131776_6_UMAP', 'decontX_GSE131776_7_UMAP

In [234]:
adata_final.obs['sample'].value_counts()

sample
GSE131776_3     2434
GSE131776_4     2370
GSE131776_16    2123
GSE131776_9     2090
GSE131776_8     2079
GSE131776_7     2005
GSE131776_6     1856
GSE131776_17    1707
GSE131776_5     1383
GSE131776_12    1371
GSE131776_11    1369
GSE131776_18    1351
GSE131776_1     1270
GSE131776_15    1123
GSE131776_2      758
GSE131776_13     667
GSE131776_10     590
GSE131776_14     529
Name: count, dtype: int64

# AAA

## GSE264071 16770 15674

In [235]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_KO.h5ad",
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_WT.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9034
Number of cells in sample Sample2: 7736


In [236]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [237]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [238]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 11:06:57 2026 .. Analyzing cells in batch 'GSE264071_KO'
Fri Apr 24 11:06:57 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 11:07:35 2026 .... Estimating contamination
Fri Apr 24 11:07:43 2026 ...... Completed iteration: 10 | converge: 0.02845
Fri Apr 24 11:07:48 2026 ...... Completed iteration: 20 | converge: 0.007031
Fri Apr 24 11:07:56 2026 ...... Completed iteration: 30 | converge: 0.003564
Fri Apr 24 11:08:02 2026 ...... Completed iteration: 40 | converge: 0.002359
Fri Apr 24 11:08:07 2026 ...... Completed iteration: 50 | converge: 0.001578
Fri Apr 24 11:08:12 2026 ...... Completed iteration: 60 | converge: 0.001012
Fri Apr 24 11:08:13 2026 ...... Completed iteration: 61 | converge: 0.0009685
Fri Apr 24 11:08:13 2026 .. Calculating final decontaminated matrix
Fri Apr 24 11:08:17 2026 .. Analyzing cells in batch 'GSE264071_WT'
Fri Apr 24 

In [239]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_merged_postR.h5ad")

In [240]:
adata_final

AnnData object with n_obs × n_vars = 16770 × 31253
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE264071_KO_UMAP', 'decontX_GSE264071_WT_UMAP'
    layers: 'decontXcounts'

In [241]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 253 个包含 'mt' 的基因


In [242]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [243]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                             n_genes_by_counts  total_counts  pct_counts_mt
AAATCCTGTAAACTGCGCCAACGATCT                652         963.0       1.349948
AAATCCTGTAAATAAATAACAGGCTCC               4169       16899.0       3.035683
AAATCCTGTAAATAAATAGGTTGGACA               1819        4375.0       3.771429
AAATCCTGTAAATAAATATCTCGGTTA                545        1204.0       1.162791
AAATCCTGTAACCAAAGTCAGGGAGGG                262         944.0      64.618644

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [244]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)
sc.pp.filter_cells(adata_final, max_genes=5000)
sc.pp.filter_genes(adata_final, min_cells=3) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 20000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 16770
Number of cells before counts filter: 16516
Number of cells beforeMT filter: 16317
Number of cells after MT filter: 15674


In [245]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 15674 × 20247
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE264071_KO_UMAP', 'decontX_GSE264071_WT_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [246]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC-R.h5ad")

In [247]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 15674 × 20247
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE264071_KO_UMAP', 'decontX_GSE264071_WT_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [248]:
adata_final.obs['sample'].value_counts()

sample
GSE264071_KO    8201
GSE264071_WT    7473
Name: count, dtype: int64

## GSE239620 92330 78250/76620

In [249]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAA.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAACD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_AAD.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-I.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-II.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-IICD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-III.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_IMH-IIICD45.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_NA.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6",
    "Sample7",
    "Sample8",
    "Sample9" 
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 7471
Number of cells in sample Sample2: 13648
Number of cells in sample Sample3: 8349
Number of cells in sample Sample4: 9617
Number of cells in sample Sample5: 6794
Number of cells in sample Sample6: 11136
Number of cells in sample Sample7: 15856
Number of cells in sample Sample8: 10675
Number of cells in sample Sample9: 8784


In [250]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [251]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [252]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 11:43:19 2026 .. Analyzing cells in batch 'GSE239620_1'
Fri Apr 24 11:43:20 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 11:44:25 2026 .... Estimating contamination
Fri Apr 24 11:44:35 2026 ...... Completed iteration: 10 | converge: 0.03082
Fri Apr 24 11:44:45 2026 ...... Completed iteration: 20 | converge: 0.01492
Fri Apr 24 11:44:54 2026 ...... Completed iteration: 30 | converge: 0.006949
Fri Apr 24 11:45:05 2026 ...... Completed iteration: 40 | converge: 0.004425
Fri Apr 24 11:45:15 2026 ...... Completed iteration: 50 | converge: 0.003023
Fri Apr 24 11:45:27 2026 ...... Completed iteration: 60 | converge: 0.002065
Fri Apr 24 11:45:39 2026 ...... Completed iteration: 70 | converge: 0.001652
Fri Apr 24 11:45:49 2026 ...... Completed iteration: 80 | converge: 0.001794
Fri Apr 24 11:45:53 2026 ...... Completed iteration: 88 | converge: 0.000

In [253]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_merged_postR.h5ad")

In [254]:
adata_final

AnnData object with n_obs × n_vars = 92330 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239620_1_UMAP', 'decontX_GSE239620_2_UMAP', 'decontX_GSE239620_3_UMAP', 'decontX_GSE239620_4_UMAP', 'decontX_GSE239620_5_UMAP', 'decontX_GSE239620_6_UMAP', 'decontX_GSE239620_7_UMAP', 'decontX_GSE239620_8_UMAP', 'decontX_GSE239620_9_UMAP'
    layers: 'decontXcounts'

In [255]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [256]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [257]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACAGCAGGAT-1               1878        5248.0       3.010671
AAACCCAGTAGGCAAC-1               1537        4473.0       2.481556
AAACCCAGTTTCTATC-1                463         947.0      24.076030
AAACCCATCTCACTCG-1               1136        2969.0       6.433142
AAACGAAGTATCCCAA-1               2601       12719.0       2.350814

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [258]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=4500) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 1000)
sc.pp.filter_cells(adata_final, max_counts = 45000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 92330
Number of cells before counts filter: 84291
Number of cells beforeMT filter: 80804
Number of cells after MT filter: 76620


In [259]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 76620 × 22203
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239620_1_UMAP', 'decontX_GSE239620_2_UMAP', 'decontX_GSE239620_3_UMAP', 'decontX_GSE239620_4_UMAP', 'decontX_GSE239620_5_UMAP', 'decontX_GSE239620_6_UMAP', 'decontX_GSE239620_7_UMAP', 'decontX_GSE239620_8_UMAP', 'decontX_GSE239620_9_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [260]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC-R.h5ad")

In [261]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 76620 × 22203
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE239620_1_UMAP', 'decontX_GSE239620_2_UMAP', 'decontX_GSE239620_3_UMAP', 'decontX_GSE239620_4_UMAP', 'decontX_GSE239620_5_UMAP', 'decontX_GSE239620_6_UMAP', 'decontX_GSE239620_7_UMAP', 'decontX_GSE239620_8_UMAP', 'decontX_GSE239620_9_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [262]:
adata_final.obs['sample'].value_counts()

sample
GSE239620_2    12824
GSE239620_7    10796
GSE239620_6    10358
GSE239620_8    10177
GSE239620_4     7893
GSE239620_3     6667
GSE239620_9     6629
GSE239620_1     5775
GSE239620_5     5501
Name: count, dtype: int64

## GSE231306 27865 24328

In [263]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_AAA.h5ad",
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_CTR.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 14086
Number of cells in sample Sample2: 13779


In [264]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [265]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [266]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 12:11:37 2026 .. Analyzing cells in batch 'GSE231306_1'
Fri Apr 24 12:11:37 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 12:12:13 2026 .... Estimating contamination
Fri Apr 24 12:12:20 2026 ...... Completed iteration: 10 | converge: 0.02815
Fri Apr 24 12:12:27 2026 ...... Completed iteration: 20 | converge: 0.007083
Fri Apr 24 12:12:33 2026 ...... Completed iteration: 30 | converge: 0.004089
Fri Apr 24 12:12:40 2026 ...... Completed iteration: 40 | converge: 0.003075
Fri Apr 24 12:12:47 2026 ...... Completed iteration: 50 | converge: 0.002149
Fri Apr 24 12:12:53 2026 ...... Completed iteration: 60 | converge: 0.001836
Fri Apr 24 12:13:00 2026 ...... Completed iteration: 70 | converge: 0.001325
Fri Apr 24 12:13:06 2026 ...... Completed iteration: 80 | converge: 0.001473
Fri Apr 24 12:13:10 2026 ...... Completed iteration: 85 | converge: 0.00

In [267]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_merged_postR.h5ad")

In [268]:
adata_final

AnnData object with n_obs × n_vars = 27865 × 55450
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE231306_1_UMAP', 'decontX_GSE231306_2_UMAP'
    layers: 'decontXcounts'

In [269]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 277 个包含 'mt' 的基因


In [270]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [271]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGACCATTC-1               4355       15410.0       1.421155
AAACCCAAGCGCCCAT-1                932        1783.0       1.346046
AAACCCAAGCTCGACC-1               1672        3768.0       0.610403
AAACCCAAGTCGAAGC-1               1479        2806.0       5.951532
AAACCCACAATATCCG-1               3942       15313.0       2.220336

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


In [272]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300) 
sc.pp.filter_cells(adata_final, max_genes=5000) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 30000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 15] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 27865


Number of cells before counts filter: 26231
Number of cells beforeMT filter: 26223
Number of cells after MT filter: 24328


In [273]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 24328 × 28549
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE231306_1_UMAP', 'decontX_GSE231306_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [274]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE231306/GSE231306_postQC-R.h5ad")

In [275]:
adata_final.obs['sample'].value_counts()

sample
GSE231306_1    12305
GSE231306_2    12023
Name: count, dtype: int64

## GSE253247 41315 40242

In [276]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P1.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P2.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_P3.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V1.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V2.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_V3.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3",
    "Sample4",
    "Sample5",
    "Sample6"
    
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()
    adata_list.append(adata_sample)
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample1: 2312


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample2: 9384
Number of cells in sample Sample3: 9581


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample4: 8662


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Number of cells in sample Sample5: 4962
Number of cells in sample Sample6: 6414


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [277]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [278]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [279]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 13:54:52 2026 .. Analyzing cells in batch 'GSE253247_1'
Fri Apr 24 13:54:52 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 13:55:04 2026 .... Estimating contamination
Fri Apr 24 13:55:05 2026 ...... Completed iteration: 10 | converge: 0.02507
Fri Apr 24 13:55:06 2026 ...... Completed iteration: 20 | converge: 0.008917
Fri Apr 24 13:55:07 2026 ...... Completed iteration: 30 | converge: 0.005097
Fri Apr 24 13:55:08 2026 ...... Completed iteration: 40 | converge: 0.003381
Fri Apr 24 13:55:09 2026 ...... Completed iteration: 50 | converge: 0.002753
Fri Apr 24 13:55:10 2026 ...... Completed iteration: 60 | converge: 0.002309
Fri Apr 24 13:55:10 2026 ...... Completed iteration: 70 | converge: 0.001968
Fri Apr 24 13:55:11 2026 ...... Completed iteration: 80 | converge: 0.001712
Fri Apr 24 13:55:12 2026 ...... Completed iteration: 90 | converge: 0.00

In [280]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_merged_postR.h5ad")

In [281]:
adata_final

AnnData object with n_obs × n_vars = 41315 × 27998
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE253247_1_UMAP', 'decontX_GSE253247_2_UMAP', 'decontX_GSE253247_3_UMAP', 'decontX_GSE253247_4_UMAP', 'decontX_GSE253247_5_UMAP', 'decontX_GSE253247_7_UMAP'
    layers: 'decontXcounts'

In [282]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 231 个包含 'mt' 的基因


In [283]:

adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [284]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGAGAACAATC-1               1848        5628.0       2.043355
AAACCTGAGAAGCCCA-1                996        2630.0       2.091255
AAACCTGAGACTAGGC-1               1746        4741.0       1.603037
AAACCTGTCCCGGATG-1               1838        7970.0       3.074028
AAACGGGAGAAGGGTA-1               1533        4802.0       1.207830

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [285]:
# pp
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=300) 
sc.pp.filter_cells(adata_final, max_genes=4500) 
sc.pp.filter_genes(adata_final, min_cells=5) 

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))

sc.pp.filter_cells(adata_final, max_counts = 15000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))

adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 41315


Number of cells before counts filter: 41153
Number of cells beforeMT filter: 40350
Number of cells after MT filter: 40242


In [286]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 40242 × 15756
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE253247_1_UMAP', 'decontX_GSE253247_2_UMAP', 'decontX_GSE253247_3_UMAP', 'decontX_GSE253247_4_UMAP', 'decontX_GSE253247_5_UMAP', 'decontX_GSE253247_7_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [287]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC-R.h5ad")

In [288]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 40242 × 15756
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE253247_1_UMAP', 'decontX_GSE253247_2_UMAP', 'decontX_GSE253247_3_UMAP', 'decontX_GSE253247_4_UMAP', 'decontX_GSE253247_5_UMAP', 'decontX_GSE253247_7_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [289]:
adata_final.obs['sample'].value_counts()

sample
GSE253247_3    9378
GSE253247_2    9140
GSE253247_4    8477
GSE253247_7    6205
GSE253247_5    4807
GSE253247_1    2235
Name: count, dtype: int64

## GSE233625 41941 34100/35534

In [290]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_STING.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_WTC.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_WTC2000.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 13180
Number of cells in sample Sample2: 16053
Number of cells in sample Sample3: 12708


In [291]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [292]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [293]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 14:34:25 2026 .. Analyzing cells in batch 'GSE233625_1'
Fri Apr 24 14:34:26 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 14:35:02 2026 .... Estimating contamination
Fri Apr 24 14:35:11 2026 ...... Completed iteration: 10 | converge: 0.03602
Fri Apr 24 14:35:19 2026 ...... Completed iteration: 20 | converge: 0.01489
Fri Apr 24 14:35:27 2026 ...... Completed iteration: 30 | converge: 0.009249
Fri Apr 24 14:35:35 2026 ...... Completed iteration: 40 | converge: 0.006368
Fri Apr 24 14:35:42 2026 ...... Completed iteration: 50 | converge: 0.004499
Fri Apr 24 14:35:50 2026 ...... Completed iteration: 60 | converge: 0.003505
Fri Apr 24 14:35:58 2026 ...... Completed iteration: 70 | converge: 0.002793
Fri Apr 24 14:36:05 2026 ...... Completed iteration: 80 | converge: 0.002322
Fri Apr 24 14:36:13 2026 ...... Completed iteration: 90 | converge: 0.001

In [294]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_merged_postR.h5ad")

In [295]:
adata_final

AnnData object with n_obs × n_vars = 41941 × 32285
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP'
    layers: 'decontXcounts'

In [296]:
adata_final.var

""
ENSMUSG00000051951
ENSMUSG00000089699
ENSMUSG00000102331
ENSMUSG00000102343
ENSMUSG00000025900
...
ENSMUSG00000095523
ENSMUSG00000095475
ENSMUSG00000094855
ENSMUSG00000095019


In [297]:
##基因id→基因名称
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
adata_final.var['original_ensembl_ids'] = adata_final.var_names

adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

In [298]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 265 个包含 'mt' 的基因


In [299]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [300]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGGATTTAG-1               3160       10056.0       4.494829
AAACCCACAACCGCCA-1               5811       33594.0       4.155504
AAACCCACAAGTATCC-1               2876       10363.0       6.272315
AAACCCACAATACAGA-1               3147       12023.0       8.142726
AAACCCACACAAGCAG-1               2386        7206.0       2.317513

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [301]:
# pp acc to author
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200) 
sc.pp.filter_cells(adata_final, max_genes=5000) 

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 41941
Number of cells beforeMT filter: 37885
Number of cells after MT filter: 35534


In [302]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 35534 × 32285
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'original_ensembl_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [303]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE233625/GSE233625_postQC-R.h5ad")

In [304]:
adata_final.obs['sample'].value_counts()

sample
GSE233625_2    14618
GSE233625_1    10708
GSE233625_3    10208
Name: count, dtype: int64

## GSE237067 17389 14586/14478

In [305]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_AAA7.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_AAA14.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_Sham.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2",
    "Sample3"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 3496
Number of cells in sample Sample2: 10108
Number of cells in sample Sample3: 3785


In [306]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [307]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [308]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 15:13:00 2026 .. Analyzing cells in batch 'GSE237067_1'
Fri Apr 24 15:13:00 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 15:13:16 2026 .... Estimating contamination
Fri Apr 24 15:13:17 2026 ...... Completed iteration: 10 | converge: 0.03127
Fri Apr 24 15:13:19 2026 ...... Completed iteration: 20 | converge: 0.00948
Fri Apr 24 15:13:20 2026 ...... Completed iteration: 30 | converge: 0.006898
Fri Apr 24 15:13:21 2026 ...... Completed iteration: 40 | converge: 0.004467
Fri Apr 24 15:13:22 2026 ...... Completed iteration: 50 | converge: 0.002977
Fri Apr 24 15:13:23 2026 ...... Completed iteration: 60 | converge: 0.002079
Fri Apr 24 15:13:25 2026 ...... Completed iteration: 70 | converge: 0.001736
Fri Apr 24 15:13:26 2026 ...... Completed iteration: 80 | converge: 0.001315
Fri Apr 24 15:13:27 2026 ...... Completed iteration: 88 | converge: 0.000

In [309]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_merged_postR.h5ad")

In [310]:
adata_final

AnnData object with n_obs × n_vars = 17389 × 32285
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP'
    layers: 'decontXcounts'

In [311]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 245 个包含 'mt' 的基因


In [312]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [313]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCACATAAGCAA-1                600        1202.0       0.332779
AAACCCACATGTCTAG-1                830        1830.0       6.830601
AAACCCACATGTTCGA-1               1092        2408.0       3.779070
AAACCCATCGAATGCT-1                325         596.0      20.973154
AAACGAAAGCGTGTTT-1                509         875.0      10.857143

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [314]:
# pp acc to author
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=220) 
sc.pp.filter_cells(adata_final, max_genes=4827) 
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 386)
sc.pp.filter_cells(adata_final, max_counts = 30284)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 20]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 17389
Number of cells before counts filter: 16986
Number of cells beforeMT filter: 16915
Number of cells after MT filter: 14478


In [315]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 14478 × 19164
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [316]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE237067/GSE237067_postQC-R.h5ad")

In [317]:
adata_final.obs['sample'].value_counts()

sample
GSE237067_2    8090
GSE237067_1    3339
GSE237067_3    3049
Name: count, dtype: int64

## GSE191226 17274 15753/15754

In [318]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_D_A.h5ad",
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_IN_A.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 9125
Number of cells in sample Sample2: 8149


In [319]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [320]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [321]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 15:23:44 2026 .. Analyzing cells in batch 'GSE191226_1'
Fri Apr 24 15:23:44 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 15:24:14 2026 .... Estimating contamination
Fri Apr 24 15:24:21 2026 ...... Completed iteration: 10 | converge: 0.02831
Fri Apr 24 15:24:28 2026 ...... Completed iteration: 20 | converge: 0.0105
Fri Apr 24 15:24:34 2026 ...... Completed iteration: 30 | converge: 0.006549
Fri Apr 24 15:24:40 2026 ...... Completed iteration: 40 | converge: 0.004927
Fri Apr 24 15:24:46 2026 ...... Completed iteration: 50 | converge: 0.003499
Fri Apr 24 15:24:52 2026 ...... Completed iteration: 60 | converge: 0.002877
Fri Apr 24 15:24:58 2026 ...... Completed iteration: 70 | converge: 0.002449
Fri Apr 24 15:25:04 2026 ...... Completed iteration: 80 | converge: 0.002009
Fri Apr 24 15:25:11 2026 ...... Completed iteration: 90 | converge: 0.0016

In [322]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_merged_postR.h5ad")

In [323]:
adata_final

AnnData object with n_obs × n_vars = 17274 × 55450
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE191226_1_UMAP', 'decontX_GSE191226_2_UMAP'
    layers: 'decontXcounts'

In [324]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 277 个包含 'mt' 的基因


In [325]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [326]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAAGAAGTGTT-1               3582        8844.0       3.165988
AAACCCAAGGGAGATA-1               4558       18401.0       1.157546
AAACCCAAGGGTCACA-1               1828        4038.0       8.915305
AAACCCACAAATTAGG-1               1372        2654.0       3.165034
AAACCCACAAGTATCC-1                432         661.0      11.649017

线粒体基因统计：
共标记了 37 个线粒体基因
这些基因是：['mt-Tf', 'mt-Rnr1', 'mt-Tv', 'mt-Rnr2', 'mt-Tl1', 'mt-Nd1', 'mt-Ti', 'mt-Tq', 'mt-Tm', 'mt-Nd2', 'mt-Tw', 'mt-Ta', 'mt-Tn', 'mt-Tc', 'mt-Ty', 'mt-Co1', 'mt-Ts1', 'mt-Td', 'mt-Co2', 'mt-Tk', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Tg', 'mt-Nd3', 'mt-Tr', 'mt-Nd4l', 'mt-Nd4', 'mt-Th', 'mt-Ts2', 'mt-Tl2', 'mt-Nd5', 'mt-Nd6', 'mt-Te', 'mt-Cytb', 'mt-Tt', 'mt-Tp']


In [327]:
# pp acc to author
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=500)  
sc.pp.filter_genes(adata_final, min_cells=3)

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, max_counts = 200000)


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 25]
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 17274


Number of cells before counts filter: 16040
Number of cells beforeMT filter: 16040
Number of cells after MT filter: 15754


In [328]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 15754 × 28536
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE191226_1_UMAP', 'decontX_GSE191226_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [329]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC-R.h5ad")

In [330]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC-R.h5ad")
adata_final

AnnData object with n_obs × n_vars = 15754 × 28536
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE191226_1_UMAP', 'decontX_GSE191226_2_UMAP'
    layers: 'raw_decontXcounts', 'uncorrected_counts'

In [331]:
adata_final.obs['sample'].value_counts()

sample
GSE191226_1    8279
GSE191226_2    7475
Name: count, dtype: int64

## GSE164678 5244 4541/4541

In [332]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_aaa.h5ad",
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_sham.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 1922
Number of cells in sample Sample2: 3322


In [333]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [334]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [335]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:1898: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 15:32:14 2026 .. Analyzing cells in batch 'GSE164678_1'
Fri Apr 24 15:32:14 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 15:32:24 2026 .... Estimating contamination
Fri Apr 24 15:32:25 2026 ...... Completed iteration: 10 | converge: 0.02478
Fri Apr 24 15:32:26 2026 ...... Completed iteration: 20 | converge: 0.007853
Fri Apr 24 15:32:27 2026 ...... Completed iteration: 30 | converge: 0.004456
Fri Apr 24 15:32:28 2026 ...... Completed iteration: 40 | converge: 0.002797
Fri Apr 24 15:32:29 2026 ...... Completed iteration: 50 | converge: 0.001845
Fri Apr 24 15:32:30 2026 ...... Completed iteration: 60 | converge: 0.001317
Fri Apr 24 15:32:31 2026 ...... Completed iteration: 67 | converge: 0.0009876
Fri Apr 24 15:32:31 2026 .. Calculating final decontaminated matrix
Fri Apr 24 15:32:32 2026 .. Analyzing cells in batch 'GSE164678_2'
Fri Apr 24 15

In [336]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_merged_postR.h5ad")

In [337]:
adata_final

AnnData object with n_obs × n_vars = 5244 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE164678_1_UMAP', 'decontX_GSE164678_2_UMAP'
    layers: 'decontXcounts'

In [338]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 238 个包含 'mt' 的基因


In [339]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [340]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                    n_genes_by_counts  total_counts  pct_counts_mt
AAACCCAGTGGTCTTA-1               2205        5311.0       1.581623
AAACCCATCGGTCTAA-1               4367       23074.0       3.042385
AAACGAAGTATGGGAC-1                763        1308.0       0.611621
AAACGAAGTCGTGTTA-1               2325        7748.0      30.562726
AAACGCTAGATGACCG-1                930        2069.0       2.126631

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Co2', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4l', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb']


In [341]:
# pp acc to author
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=200)  

print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 10] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 5244
Number of cells beforeMT filter: 5113
Number of cells after MT filter: 4541


In [342]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 4541 × 31053
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE164678_1_UMAP', 'decontX_GSE164678_2_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [343]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE164678/GSE164678_postQC-R.h5ad")

In [344]:
adata_final.obs['sample'].value_counts()

sample
GSE164678_2    2886
GSE164678_1    1655
Name: count, dtype: int64

## GSE141031 13056 13019/12496

In [345]:
file_paths = [
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_Apoe_0M_1M_2M_4M.h5ad",
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_dko_0M_1M_2M_4M.h5ad"
]

sample_ids = [
    "Sample1",
    "Sample2"
]
# 读取所有H5AD文件
adata_list = []
for path, sample_id in zip(file_paths, sample_ids):
    adata_sample = sc.read_h5ad(path)
    print("Number of cells in sample {}: {}".format(sample_id, len(adata_sample.obs)))
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {sample_id} 中缺少 'sample' 元数据列")
    adata_sample.obs_names_make_unique()
    adata_sample.var_names_make_unique()
    adata_list.append(adata_sample)
    

Number of cells in sample Sample1: 6797
Number of cells in sample Sample2: 6259


In [346]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)
adata_final.write_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged.h5ad")

In [347]:
# Start with doublet removal and ambient RNA correction
import anndata2ri
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
import scanpy as sc

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


/tmp/ipykernel_1945477/2937564426.py:10: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [348]:
%%R 
library(celda)
library(zellkonverter)
library(SummarizedExperiment)
library(scDblFinder)
sce = readH5AD("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged.h5ad")
samples = sce$sample
assays(sce)$counts <- assays(sce)$X
assays(sce)$X <- NULL
sce1 <- scDblFinder(sce, samples=samples)
sce2 <- decontX(sce1, batch = samples)
sce_adata <- writeH5AD(sce2, file="data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged_postR.h5ad")

/home/lixiangyu/.cache/R/basilisk/1.14.3/zellkonverter/1.12.1/zellkonverterAnnDataEnv-0.10.2/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


--------------------------------------------------
Starting DecontX
--------------------------------------------------
Fri Apr 24 15:36:25 2026 .. Analyzing cells in batch 'GSE141031_apoe1'
Fri Apr 24 15:36:25 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 15:36:35 2026 .... Estimating contamination
Fri Apr 24 15:36:36 2026 ...... Completed iteration: 10 | converge: 0.01407
Fri Apr 24 15:36:37 2026 ...... Completed iteration: 20 | converge: 0.00302
Fri Apr 24 15:36:38 2026 ...... Completed iteration: 30 | converge: 0.001731
Fri Apr 24 15:36:39 2026 ...... Completed iteration: 40 | converge: 0.001278
Fri Apr 24 15:36:39 2026 ...... Completed iteration: 49 | converge: 0.0009874
Fri Apr 24 15:36:39 2026 .. Calculating final decontaminated matrix
Fri Apr 24 15:36:40 2026 .. Analyzing cells in batch 'GSE141031_apoe2'
Fri Apr 24 15:36:40 2026 .... Generating UMAP and estimating cell types
Fri Apr 24 15:36:50 2026 .... Estimating contamination
Fri Apr 24 15:36:51 2026 ...... C

In [349]:
adata_final = sc.read_h5ad("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_merged_postR.h5ad")

In [350]:
adata_final.obs['sample'].value_counts()

sample
GSE141031_apoe3    2659
GSE141031_dko3     2253
GSE141031_dko4     1577
GSE141031_apoe2    1488
GSE141031_apoe1    1415
GSE141031_dko2     1302
GSE141031_apoe4    1235
GSE141031_dko1     1127
Name: count, dtype: int64

In [351]:
##基因id→基因名称
ensembl_id_df = pd.read_csv("mouse_gene_names_to_ensembl.csv")
ensembl_to_gene = dict(zip(ensembl_id_df['ensembl_id'], ensembl_id_df['gene_name']))
adata_final.var['original_ensembl_ids'] = adata_final.var_names
adata_final.var_names = [
    ensembl_to_gene[ensembl_id] if ensembl_id in ensembl_to_gene else ensembl_id 
    for ensembl_id in adata_final.var_names
]

In [352]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata_final.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata_final.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata_final.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 0 个以 'MT-' 开头的基因
找到 13 个以 'mt-' 开头的基因
找到 190 个包含 'mt' 的基因


In [353]:
adata_final.var['mt'] = adata_final.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata_final, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [354]:
# 查看计算得到的指标
print("细胞水平指标：")
print(adata_final.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].head())

# 查看线粒体基因的统计
print("\n线粒体基因统计：")
mito_genes = adata_final.var_names[adata_final.var['mt']]
print(f"共标记了 {len(mito_genes)} 个线粒体基因")
print(f"这些基因是：{mito_genes.tolist()}")

细胞水平指标：
                          n_genes_by_counts  total_counts  pct_counts_mt
AAACCTGCATGCATGT_apoe_0m               2951   4248.938212       0.912954
AAACCTGGTACCGAGA_apoe_0m               1716   2964.945262       1.395850
AAACCTGGTATATCCG_apoe_0m               3966   5405.915428       0.715048
AAACCTGGTCTAACGT_apoe_0m               3445   5066.894762       0.808658
AAACCTGGTCTTCAAG_apoe_0m               3341   4884.659199       0.807413

线粒体基因统计：
共标记了 13 个线粒体基因
这些基因是：['mt-Rnr1', 'mt-Rnr2', 'mt-Nd1', 'mt-Nd2', 'mt-Co1', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd4', 'mt-Nd5', 'mt-Nd6', 'mt-Cytb', 'mt-Tp']


In [355]:
# pp acc to author
print('Number of cells before gene filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_genes=1500)  

print('Number of cells before counts filter: {:d}'.format(adata_final.n_obs))
sc.pp.filter_cells(adata_final, min_counts = 3000) 


print('Number of cells beforeMT filter: {:d}'.format(adata_final.n_obs))
adata_final = adata_final[adata_final.obs['pct_counts_mt'] < 5] 
print('Number of cells after MT filter: {:d}'.format(adata_final.n_obs))

Number of cells before gene filter: 13056
Number of cells before counts filter: 12982
Number of cells beforeMT filter: 12496
Number of cells after MT filter: 12496


In [356]:
##保留原始计数
adata_final.layers["uncorrected_counts"] = adata_final.X.copy()
adata_final.layers["raw_decontXcounts"] = adata_final.layers["decontXcounts"].copy()
adata_final.X = np.around(adata_final.layers["raw_decontXcounts"].copy()).astype(int)
del adata_final.layers["decontXcounts"]
adata_final

/tmp/ipykernel_1945477/3519320560.py:2: ImplicitModificationWarning: Setting element `.layers['uncorrected_counts']` of view, initializing view as actual.
  adata_final.layers["uncorrected_counts"] = adata_final.X.copy()


AnnData object with n_obs × n_vars = 12496 × 11447
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'original_ensembl_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'X_name', 'decontX', 'scDblFinder.threshold'
    obsm: 'decontX_GSE141031_apoe1_UMAP', 'decontX_GSE141031_apoe2_UMAP', 'decontX_GSE141031_apoe3_UMAP', 'decontX_GSE141031_apoe4_UMAP', 'decontX_GSE141031_dko1_UMAP', 'decontX_GSE141031_dko2_UMAP', 'decontX_GSE141031_dko3_UMAP', 'decontX_GSE141031_dko4_UMAP'
    layers: 'uncorrected_counts', 'raw_decontXcounts'

In [357]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC-R.h5ad")

In [358]:
adata_final.obs['sample'].value_counts()

sample
GSE141031_apoe3    2529
GSE141031_dko3     2140
GSE141031_dko4     1533
GSE141031_apoe2    1448
GSE141031_apoe1    1378
GSE141031_dko2     1249
GSE141031_apoe4    1129
GSE141031_dko1     1090
Name: count, dtype: int64

In [359]:
sample_mapping = {
    'GSE141031_apoe1': 'GSE141031_1',
    'GSE141031_apoe2': 'GSE141031_2',
    'GSE141031_apoe3': 'GSE141031_3',
    'GSE141031_apoe4': 'GSE141031_4',
    'GSE141031_dko1': 'GSE141031_5',
    'GSE141031_dko2': 'GSE141031_6',
    'GSE141031_dko3': 'GSE141031_7',
    'GSE141031_dko4': 'GSE141031_8',
}
adata_final.obs['sample'] = adata_final.obs['sample'].astype(str)
adata_final.obs['sample'] = adata_final.obs['sample'].map(sample_mapping)

In [360]:
adata_final.obs['sample'].value_counts()

sample
GSE141031_3    2529
GSE141031_7    2140
GSE141031_8    1533
GSE141031_2    1448
GSE141031_1    1378
GSE141031_6    1249
GSE141031_4    1129
GSE141031_5    1090
Name: count, dtype: int64

In [361]:
adata_final.write("data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC-R.h5ad")

# merged

In [4]:
file_paths = [
    "data/Mouse_public_data/GSE239591/GSE239591_postQC-R.h5ad",
    "data/Mouse_public_data/GSE269449/GSE269449_postQC-R.h5ad",
    "data/Mouse_public_data/GSE298574/GSE298574_postQC-R.h5ad",
    "data/Mouse_public_data/GSE274572/GSE274572_postQC-R.h5ad",
    "data/Mouse_public_data/GSE210406/GSE210406_postQC-R.h5ad",
    "data/Mouse_public_data/GSE262694/GSE262694_postQC-R.h5ad",
    "data/Mouse_public_data/GSE284253/GSE284253_postQC-R.h5ad",
    "data/Mouse_public_data/GSE279601/GSE279601_postQC-R.h5ad",
    "data/Mouse_public_data/GSE211216/GSE211216_postQC-R.h5ad",
    "data/Mouse_public_data/GSE245373/GSE245373_postQC_mt-R.h5ad",
    "data/Mouse_public_data/GSE264253/GSE264253_postQC-R.h5ad",
    "data/Mouse_public_data/GSE209525/GSE209525_postQC-R.h5ad",
    "data/Mouse_public_data/GSE205930/GSE205930_postQC-R.h5ad",
    "data/Mouse_public_data/GSE155513/GSE155513_postQC-R.h5ad",
    "data/Mouse_public_data/GSE134557/GSE134557_postQC-R.h5ad",
    "data/Mouse_public_data/GSE131776/GSE131776_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE141031/GSE141031_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE164678/GSE164678_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE191226/GSE191226_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE231306/GSE231306_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE233625/GSE233625_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE237067/GSE237067_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE239620/GSE239620_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE253247/GSE253247_postQC-R.h5ad",
    "data/Mouse_public_data/exp_AS/GSE264071/GSE264071_postQC-R.h5ad"
]

datasets = [
    "Dataset1",
    "Dataset2",
    "Dataset3",
    "Dataset4",
    "Dataset5",
    "Dataset6",
    "Dataset7",
    "Dataset8",
    "Dataset9",
    "Dataset10",
    "Dataset11",
    "Dataset12",
    "Dataset13",
    "Dataset14",
    "Dataset15",
    "Dataset16",
    "Dataset17",
    "Dataset18",
    "Dataset19",
    "Dataset20",
    "Dataset21",
    "Dataset22",
    "Dataset23",
    "Dataset24",
    "Dataset25"
]
# 读取所有H5AD文件
adata_list = []
for path, datasets in zip(file_paths, datasets):
    adata_sample = sc.read_h5ad(path)
    
    print("Number of cells in sample {}: {}".format(datasets, len(adata_sample.obs)))
    
    if 'sample' not in adata_sample.obs.columns:
        print(f"警告: 样本 {datasets} 中缺少 'sample' 元数据列")
    
    adata_list.append(adata_sample)

Number of cells in sample Dataset1: 28114
Number of cells in sample Dataset2: 10513
Number of cells in sample Dataset3: 26239
Number of cells in sample Dataset4: 7525
Number of cells in sample Dataset5: 14231
Number of cells in sample Dataset6: 80165
Number of cells in sample Dataset7: 4961
Number of cells in sample Dataset8: 15836
Number of cells in sample Dataset9: 6124
Number of cells in sample Dataset10: 5821
Number of cells in sample Dataset11: 7156
Number of cells in sample Dataset12: 15387
Number of cells in sample Dataset13: 38460
Number of cells in sample Dataset14: 42372
Number of cells in sample Dataset15: 2439
Number of cells in sample Dataset16: 27075
Number of cells in sample Dataset17: 12496
Number of cells in sample Dataset18: 4541
Number of cells in sample Dataset19: 15754
Number of cells in sample Dataset20: 24328
Number of cells in sample Dataset21: 35534
Number of cells in sample Dataset22: 14478
Number of cells in sample Dataset23: 76620
Number of cells in sample D

In [5]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata_final.write_h5ad("./ANNOTATE_new/7_mapping_new_data/output_all_mouse260420/Mouse_merged.h5ad")

In [7]:
adata_final

AnnData object with n_obs × n_vars = 572085 × 83919
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    obsm: 'decontX_GSE239591_1_UMAP', 'decontX_GSE239591_2_UMAP', 'decontX_GSE239591_3_UMAP', 'decontX_GSE239591_4_UMAP', 'decontX_GSE239591_5_UMAP', 'decontX_GSE239591_6_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE298574_1_UMAP', 'decontX_GSE298574_2_UMAP', 'decontX_GSE274572_1_UMAP', 'decontX_GSE274572_2_UMAP', 'decontX_GSE210406_1_UMAP', 'decontX_GSE210406_2_UMAP', 'decontX_GSE210406_3_UMAP', 'decontX_GSE210406_4_UMAP', 'decontX_GSE262694_1_UMAP', 'decontX_GSE262694_2_UMAP', 'decontX_GSE262694_3_UMAP', 'decontX_GSE262694_4_UMAP', 'decontX_GSE262694_5_UMAP', 'decontX_

In [222]:
#####################################37个基因都来自GSE274572 GSE210406#####################################

In [223]:
adata= sc.read_h5ad("output_data/public_data/Mouse_AS/Mouse_merged_mt.h5ad")

KeyboardInterrupt: 

In [322]:
# 筛选以"MT-"开头的基因
mt_genes_upper = [gene for gene in adata.var_names if gene.startswith('MT-')]
# 筛选以"mt-"开头的基因
mt_genes_lower = [gene for gene in adata.var_names if gene.startswith('mt-')]
# 筛选包含"mt"的基因
mt_genes_any = [gene for gene in adata.var_names if 'mt' in gene.lower()]
print(f"找到 {len(mt_genes_upper)} 个以 'MT-' 开头的基因")
print(f"找到 {len(mt_genes_lower)} 个以 'mt-' 开头的基因")
print(f"找到 {len(mt_genes_any)} 个包含 'mt' 的基因")

找到 13 个以 'MT-' 开头的基因
找到 37 个以 'mt-' 开头的基因
找到 499 个包含 'mt' 的基因


In [323]:
mt_genes_lower

['mt-Atp6',
 'mt-Atp8',
 'mt-Co1',
 'mt-Co2',
 'mt-Co3',
 'mt-Cytb',
 'mt-Nd1',
 'mt-Nd2',
 'mt-Nd3',
 'mt-Nd4',
 'mt-Nd4l',
 'mt-Nd5',
 'mt-Nd6',
 'mt-Rnr1',
 'mt-Rnr2',
 'mt-Ta',
 'mt-Tc',
 'mt-Td',
 'mt-Te',
 'mt-Tf',
 'mt-Tg',
 'mt-Th',
 'mt-Ti',
 'mt-Tk',
 'mt-Tl1',
 'mt-Tl2',
 'mt-Tm',
 'mt-Tn',
 'mt-Tp',
 'mt-Tq',
 'mt-Tr',
 'mt-Ts1',
 'mt-Ts2',
 'mt-Tt',
 'mt-Tv',
 'mt-Tw',
 'mt-Ty']